In [1]:
import os
import sqlite3
import pandas as pd
from tqdm import tqdm
from langchain_community.document_loaders import CSVLoader #
from langchain_core.documents import Document #
from langchain_text_splitters import RecursiveCharacterTextSplitter #
from langchain_ollama import OllamaEmbeddings #
from langchain_chroma import Chroma #


# VECTORIZE

## ADDRESS PREP (DATA SOURCE 1)

In [2]:
location_mukim_district_state = pd.read_csv('../output/malaysia-postcodes-location-mukim-district-state.csv',dtype='string')
location_mukim_district_state['address'] = location_mukim_district_state.apply(
    lambda row: f"{row['location']}, {row['mukim']}, {row['postcode']}, {row['district']}, {row['state']}", axis=1
) 

location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: x.split(', '))
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: [i for i in x if i != '<NA>'])
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: list(dict.fromkeys(x)))
location_mukim_district_state['address'] = location_mukim_district_state['address'].apply(lambda x: ', '.join(x))
address=location_mukim_district_state[['address']]
address.to_csv('../output/address_src_1.csv', index=False)

## ADDRESS PREP (DATA SOURCE 2)

In [11]:
locations_osm = pd.read_csv('../data_source/processed_my_osm.csv',dtype='string')
locations_osm['address'] = locations_osm['address'].str.upper()
address = locations_osm[['address']]
address.to_csv('../output/address_src_2.csv', index=False)

## ADDRESS PREP (DATA SOURCE 3)

In [18]:
locations_school = pd.read_csv('../data_source/school.csv',dtype='string')
locations_school = locations_school[['ALAMATSURAT','POSKODSURAT','BANDARSURAT','NEGERI']]

locations_school['address'] = locations_school.apply(
    lambda row: f"{row['ALAMATSURAT']}, {row['POSKODSURAT']}, {row['BANDARSURAT']}, {row['NEGERI']}", axis=1
) 

address = locations_school[['address']]
address.to_csv('../output/address_src_3.csv', index=False)

## ADDRESS EMBEDDING

#### Source 1

In [3]:
file_path_csv1 = '../output/address_src_1.csv'

# Create a CSVLoader instance
loader1 = CSVLoader(file_path=file_path_csv1)
loader1

#### Source 2

In [12]:
file_path_csv2 = '../output/address_src_2.csv'

# Create a CSVLoader instance
loader2 = CSVLoader(file_path=file_path_csv2)
loader2

### Source 3

In [19]:
file_path_csv3 = '../output/address_src_3.csv'

# Create a CSVLoader instance
loader3 = CSVLoader(file_path=file_path_csv3)
loader3

### CREATE DOCUMENT

#### Source 1

In [4]:
import numpy as np

# Load documents1 from CSV file without column headers
documents1 = loader1.load()
documents1

# Remode string "address: " from the documents1 and maintain metadata
documents1 = [Document(page_content=doc.page_content.replace('address: ', ''), metadata=doc.metadata) for doc in documents1]

# Update metadata state by using the last part of the address
for doc in documents1:
    address_parts = doc.page_content.split(', ')
    if address_parts:
        doc.metadata['state'] = address_parts[-1]
    if len(address_parts) == 4:
        doc.metadata['district'] = address_parts[-2]
        doc.metadata['postcode'] = address_parts[-3]
        doc.metadata['city'] = np.nan
    else:
        doc.metadata['district'] = np.nan
        doc.metadata['postcode'] = np.nan
        doc.metadata['city'] = np.nan

#documents1

In [5]:
documents1[0]

Document(metadata={'source': '../output/address_src_1.csv', 'row': 0, 'state': 'PERLIS', 'district': 'KANGAR', 'postcode': '01000', 'city': nan}, page_content='ABI, 01000, KANGAR, PERLIS')

In [6]:
documents1[0].page_content[:1000]  # Display the first 1000 characters of the first document


'ABI, 01000, KANGAR, PERLIS'

In [7]:
print(len(documents1))
total_docs  = len(documents1)

58394


In [8]:
from sentence_transformers import SentenceTransformer
from langchain.embeddings import HuggingFaceEmbeddings

# Initialize the embedding
oembed = OllamaEmbeddings(base_url="http://localhost:11434", model="llama3.2:latest") # 3072-dim
hfembed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")  # 384-dim

c:\Users\izard\miniconda3\envs\etl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\izard\AppData\Local\Temp\ipykernel_6040\2698455389.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hfembed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")  # 384-dim


In [9]:
# Define the folder path for Chroma's in-memory storage
persist_directory = "../output/vectorstore"

In [ ]:
'''
for i in tqdm(range(0, len(documents))):
    vectorstore = Chroma.from_documents(
        documents=[documents[i]], 
        embedding=oembed, 
        persist_directory=persist_directory,
        collection_name="base_address"  # Specify the collection name here
    )
print('Data Ingested into Vectorstore')
'''

In [10]:
chunk_size = 100
for i in tqdm(range(0, len(documents1), chunk_size)):
    batch = documents1[i:i+chunk_size]
    Chroma.from_documents(
        documents=batch,
        embedding=hfembed,
        persist_directory=persist_directory,
        collection_name="base_address"
    )


100%|██████████| 584/584 [13:13<00:00,  1.36s/it]


#### Source 2

In [13]:
import numpy as np 
# Load documents2 from CSV file without column headers
documents2 = loader2.load()
documents2

# Remode string "address: " from the documents2 and maintain metadata
documents2 = [Document(page_content=doc.page_content.replace('address: ', ''), metadata=doc.metadata) for doc in documents2]

# Update metadata state by using the last part of the address
for doc in documents2:
    address_parts = doc.page_content.split(', ')
    if address_parts:
        doc.metadata['state'] = address_parts[-1]
    if len(address_parts) == 6:
        doc.metadata['district'] = address_parts[-2]
        doc.metadata['postcode'] = address_parts[-3]
        doc.metadata['city'] = address_parts[-4]
    else:
        doc.metadata['district'] = np.nan
        doc.metadata['postcode'] = np.nan
        doc.metadata['city'] = np.nan
        
#documents2

In [14]:
documents2[0].page_content[:1000] 

'433, JALAN 5/46, PETALING JAYA, 46000, PETALING, SELANGOR'

In [15]:
print(len(documents2))
total_docs  = len(documents2)

65395


In [16]:
# Define the folder path for Chroma's in-memory storage
persist_directory = "../output/vectorstore"

In [17]:
chunk_size = 100
for i in tqdm(range(0, len(documents2), chunk_size)):
    batch = documents2[i:i+chunk_size]
    Chroma.from_documents(
        documents=batch,
        embedding=hfembed,
        persist_directory=persist_directory,
        collection_name="base_address"
    )


100%|██████████| 654/654 [20:09<00:00,  1.85s/it]


### Source 3

In [21]:
import numpy as np 
# Load documents3 from CSV file without column headers
documents3 = loader3.load()
documents3

# Remode string "address: " from the documents3 and maintain metadata
documents3 = [Document(page_content=doc.page_content.replace('address: ', ''), metadata=doc.metadata) for doc in documents3]

# Update metadata state by using the last part of the address
for doc in documents3:
    address_parts = doc.page_content.split(', ')
    if address_parts:
        doc.metadata['state'] = address_parts[-1]
    if len(address_parts) == 4:
        doc.metadata['district'] = address_parts[-2]
        doc.metadata['postcode'] = address_parts[-3]
        doc.metadata['city'] = np.nan
    else:
        doc.metadata['district'] = np.nan
        doc.metadata['postcode'] = np.nan
        doc.metadata['city'] = np.nan
        
#documents3

In [22]:
documents3[0].page_content[:1000] 

'JALAN KELAB, 35000, TAPAH, PERAK'

In [23]:
print(len(documents3))
total_docs  = len(documents3)

10201


In [24]:
chunk_size = 100
for i in tqdm(range(0, len(documents3), chunk_size)):
    batch = documents3[i:i+chunk_size]
    Chroma.from_documents(
        documents=batch,
        embedding=hfembed,
        persist_directory=persist_directory,
        collection_name="base_address"
    )


100%|██████████| 103/103 [03:37<00:00,  2.11s/it]
